In [21]:
import pandas as pd
import torch
from sklearn.metrics import classification_report
from transformers import AutoTokenizer, AutoModelForSequenceClassification

In [22]:
df = pd.read_csv("data/sentiment_dataset.csv")
df["label"] = df["label"].map({0: "Нейтральная", 1: "Позитивная", 2: "Негативная"})
df.head()

,text,label,src
0,"Пальто красивое, но пришло с дырой в молнии. П...",Нейтральная,rureviews
1,"Очень долго шел заказ,ждала к новому году,приш...",Нейтральная,rureviews
2,"Могу сказать одно, брюки нормальные, НО они бы...",Нейтральная,rureviews
3,"Доставка быстрая, меньше месяца. Заказывали ра...",Нейтральная,rureviews
4,Мне не очень понравилось это платье. Размер ...,Нейтральная,rureviews


In [23]:
model_name = "blanchefort/rubert-base-cased-sentiment"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

C:\My\PythonProjects\ML_edu\.venv\Lib\site-packages\huggingface_hub\file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [24]:
df = df.sample(n=100, random_state=97)

@torch.no_grad()
def predict_sentiment(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
    outputs = model(**inputs)
    probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
    labels = ["Нейтральная", "Позитивная", "Негативная"]
    pred_idx = torch.argmax(probs, dim=1).item()
    return labels[pred_idx]

df["predicted_label"] = df["text"].apply(predict_sentiment)

In [25]:
df

,text,label,src,predicted_label
118503,Фейт я начинал смотреть с найта и тогда он мне...,Негативная,anime,Позитивная
146867,Отличный центр! Пришел на лечение по квоте и в...,Позитивная,geo,Позитивная
32667,Юбка просто класс!,Позитивная,rureviews,Позитивная
271396,"Много жира и жил, еле порезала. Совсем на фото...",Нейтральная,perekrestok,Нейтральная
246537,Этот продукт бужениной называть нельзя. Это ва...,Негативная,perekrestok,Негативная
...,...,...,...,...
58682,Очень долго не могли отправить. Нет там\r\n ст...,Негативная,rureviews,Негативная
283751,"Достоинства:\nОчень вкусная, два батончика в у...",Позитивная,perekrestok,Нейтральная
59667,"Мало, подарила девочке подростку. изначально д...",Негативная,rureviews,Негативная
150406,"Первый раз пользовалась этой услугой, однако, ...",Позитивная,geo,Позитивная


In [26]:
print(classification_report(df["label"], df["predicted_label"]))

              precision    recall  f1-score   support

  Негативная       0.53      0.68      0.59        28
 Нейтральная       0.63      0.46      0.53        37
  Позитивная       0.68      0.71      0.69        35

    accuracy                           0.61       100
   macro avg       0.61      0.62      0.61       100
weighted avg       0.62      0.61      0.61       100



In [32]:
def predict(csv_file):
    fdf = pd.read_csv(csv_file, sep=",", quotechar='"')
    fdf["ТОНАЛЬНОСТЬ"] = fdf["ТЕКСТ ПОСТА"].apply(predict_sentiment)
    new_name = f"{csv_file.replace('csv', 'out').split('.')[0]}_С_Тональностью.csv"
    fdf.to_csv(new_name, index=False, encoding="utf-8-sig")
    print(f"Сохранен файл {new_name}")

In [33]:
from converter import OUTPUT_FILES as FILES

for file in FILES:
    predict(file)

Сохранен файл data/out/Москва_Мое_сокровище_С_Тональностью.csv
Сохранен файл data/out/Московская_область_С_Тональностью.csv
Сохранен файл data/out/Подарок_малышу_Челябинск_С_Тональностью.csv
Сохранен файл data/out/Татарстан_С_Тональностью.csv
Сохранен файл data/out/Ульяновская_область_С_Тональностью.csv


FileNotFoundError: [Errno 2] No such file or directory: 'data/csv/Шкатулка_Расту_в_Югре.csv'